# One grouped training session
CPU-verify the 42M finalist, optimizer, data and tests. Then one G4 session measures training batch speed, checks a changed batch on short equal-example pilots, and continues to a fixed total example budget (20k updates at batch 128). Every checkpoint is saved to Drive with exact predictions. Maximum 45 minutes; no full-corpus run.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU first'
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
repo=Path('/content/hse-nlp-grouped-training')
subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.grouped_training import prepare
prepare()


Export and verify prepared/grouped-42m-20k-v1/ready.json from Drive. Then select a BF16-capable GPU with at least 40 GiB memory. Do not Run All.


In [ ]:
from google.colab import drive, runtime
try:
    drive.mount('/content/drive', timeout_ms=90000)
    import os, sys, json, hashlib, tarfile, subprocess, asyncio, signal, time
    from pathlib import Path
    prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/grouped-42m-20k-v1')
    ready=json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_verified']
    payload=(prepared/'source.tar').read_bytes()
    assert hashlib.sha256(payload).hexdigest()==ready['source_sha256']
    work=Path('/content/hw2-grouped-training')
    work.mkdir(exist_ok=False)
    os.chdir(work)
    Path('source.tar').write_bytes(payload)
    with tarfile.open('source.tar') as archive: archive.extractall('.',filter='data')
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True,timeout=180)
    os.environ['USE_TF']='0'
    os.environ['USE_FLAX']='0'
    pilot_log=(prepared/'execution.log').open('w')
    pilot_process=subprocess.Popen([sys.executable,'-u','-m','homeworks.hw2.grouped_training','--gpu'],stdout=pilot_log,stderr=subprocess.STDOUT,start_new_session=True)
    async def finish_pilots():
        status={'pid':pilot_process.pid,'started_utc':time.time()}
        try:
            await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=2700)
            status['returncode']=pilot_process.returncode
        except BaseException as error:
            status['error']=type(error).__name__+': '+str(error)
            if pilot_process.poll() is None:
                os.killpg(pilot_process.pid,signal.SIGTERM)
                try: await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=10)
                except asyncio.TimeoutError: os.killpg(pilot_process.pid,signal.SIGKILL)
        finally:
            pilot_log.close()
            status['finished_utc']=time.time()
            try: (prepared/'execution.complete.json').write_text(json.dumps(status,indent=2))
            finally: runtime.unassign()
    pilot_task=asyncio.create_task(finish_pilots())
    print('GROUPED_TRAINING_STARTED',pilot_process.pid,'Automatic notebook-kernel release is armed.')
except BaseException:
    runtime.unassign()
    raise


Monitor results and export each checkpoint from Drive for bot evaluation. Save its receipt beside the exact JSONL. After completion verify actual GPU disconnection; manually release if needed. The driver never resumes a failed run or overwrites a checkpoint automatically.
